<a href="https://colab.research.google.com/github/Prashant-0809/Data-privacy-/blob/main/Information_Security_practicals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cipher Implementations Explained

This notebook contains implementations of several classical ciphers. Here's a brief explanation of how each one works:

## 1. Caesar Cipher (Substitution)
*   **Concept**: This is one of the simplest and most widely known encryption techniques. It's a type of substitution cipher in which each letter in the plaintext is replaced by a letter some fixed number of positions down the alphabet.
*   **Encryption**: For a given `shift` value, each letter is moved forward in the alphabet by that many positions. For example, with a shift of 3, 'A' becomes 'D', 'B' becomes 'E', and so on.
*   **Decryption**: To decrypt, the process is simply reversed, shifting each letter backward by the same `shift` value.

## 2. Monoalphabetic Cipher (Substitution)
*   **Concept**: In this cipher, each letter in the plaintext is mapped to a *different, unique* letter in the ciphertext. Unlike Caesar, the substitution is not a simple shift but can be arbitrary.
*   **Key Generation**: A `keyword` is used to create a shuffled alphabet (the `key_map`). The keyword letters come first (deduplicated), followed by the remaining letters of the alphabet in order.
*   **Encryption**: Each letter of the plaintext is replaced by its corresponding letter from the generated `key_map`.
*   **Decryption**: A reverse `key_map` is used to convert ciphertext letters back to their original plaintext letters.

## 3. Vigenère Cipher (Polyalphabetic Substitution)
*   **Concept**: An improvement over the monoalphabetic ciphers, the Vigenère cipher uses a series of different Caesar ciphers based on the letters of a keyword.
*   **Encryption**: The `key` is repeated to match the length of the plaintext. Each letter of the plaintext is then shifted by an amount determined by the corresponding letter of the key (e.g., 'A' for 0 shift, 'B' for 1 shift, etc.).
*   **Decryption**: Each ciphertext letter is shifted backward by the amount determined by the corresponding key letter.

## 4. Playfair Cipher (Substitution)
*   **Concept**: This is the first practical digraphic cipher, encrypting pairs of letters (digrams) instead of single letters. It uses a 5x5 key matrix derived from a keyword.
*   **Key Matrix Generation**: A 5x5 grid is filled with the letters of the `key` (excluding duplicates and treating 'I' and 'J' as the same), followed by the remaining letters of the alphabet.
*   **Preparation**: Plaintext is converted into pairs of letters. If a pair has two identical letters (e.g., 'LL'), an 'X' is inserted between them. If the plaintext has an odd number of letters, an 'X' is appended at the end.
*   **Encryption/Decryption Rules**: Digrams are encrypted based on their positions in the key matrix:
    *   If letters are in the same row, shift right (encrypt) or left (decrypt).
    *   If letters are in the same column, shift down (encrypt) or up (decrypt).
    *   If letters form a rectangle, swap corners (e.g., top-left and bottom-right swap with top-right and bottom-left).

## 5. Hill Cipher (Substitution)
*   **Concept**: A polygraphic substitution cipher based on linear algebra, encrypting blocks of letters using matrix multiplication over a modulo 26 arithmetic system.
*   **Key**: A square matrix (`key_matrix`) whose determinant must be coprime to 26 (to ensure invertibility).
*   **Encryption**: Plaintext is divided into blocks of size `n` (where `n` is the dimension of the key matrix). Each block is represented as a column vector of numbers (A=0, B=1, ...). This vector is multiplied by the `key_matrix`, and the result is taken modulo 26 to get the ciphertext block.
*   **Decryption**: Requires calculating the modular inverse of the `key_matrix` (modulo 26). The ciphertext blocks are then multiplied by this inverse matrix to recover the plaintext.

## 6. Rail Fence Cipher (Transposition)
*   **Concept**: A simple transposition cipher that writes plaintext in a zigzag pattern across a number of "rails" and then reads off the letters row by row.
*   **Encryption**: The plaintext letters are written diagonally down and up across a specified number of `rails`. The ciphertext is then formed by reading the letters from each rail sequentially.
*   **Decryption**: The algorithm reconstructs the rail pattern, determines the positions of the ciphertext letters in the original grid, and then reads them in zigzag order to recover the plaintext.

## 7. Row Transposition Cipher (Transposition)
*   **Concept**: Rearranges the letters of the plaintext by writing them into a grid and then reading the columns in an order specified by a keyword.
*   **Encryption**: Plaintext is written into a grid, row by row. The columns are then reordered based on the alphabetical order of the letters in the `key`. The ciphertext is read column by column in the new order.
*   **Decryption**: The grid dimensions are recalculated. The ciphertext is split into columns, and these columns are reassembled into the original order using the key's alphabetical sequence. The plaintext is then read row by row.

## 8. Product Cipher (Vigenère + Row Transposition)
*   **Concept**: Combines two or more ciphers in sequence to achieve a stronger encryption. This implementation uses Vigenère (substitution) followed by Row Transposition (transposition).
*   **Encryption**: First, the plaintext is encrypted using the Vigenère cipher. The output of this step is then used as the input for the Row Transposition cipher, producing the final ciphertext.
*   **Decryption**: The decryption process reverses the steps: first, the Row Transposition cipher is decrypted, and then the result is decrypted using the Vigenère cipher to obtain the original plaintext.

In [1]:
import numpy as np
import string

# 1. CAESAR CIPHER  (Substitution)
# Shifts every letter by a fixed number of positions in the alphabet.

def caesar_encrypt(plaintext: str, shift: int) -> str:
    result = []
    for ch in plaintext.upper():
        if ch.isalpha():
            result.append(chr((ord(ch) - ord('A') + shift) % 26 + ord('A')))
        else:
            result.append(ch)
    return ''.join(result)

def caesar_decrypt(ciphertext: str, shift: int) -> str:
    return caesar_encrypt(ciphertext, -shift)


# =============================================================================
# 2. MONOALPHABETIC CIPHER  (Substitution)
# =============================================================================
# Every letter maps to a fixed, unique substitute from a shuffled alphabet.

def mono_generate_key(keyword: str) -> dict:
    """Build a substitution alphabet from a keyword."""
    keyword   = ''.join(dict.fromkeys(keyword.upper()))          # deduplicate
    remaining = [c for c in string.ascii_uppercase if c not in keyword]
    cipher_alpha = keyword + ''.join(remaining)
    key_map      = {string.ascii_uppercase[i]: cipher_alpha[i] for i in range(26)}
    return key_map

def mono_encrypt(plaintext: str, key_map: dict) -> str:
    return ''.join(key_map.get(c, c) for c in plaintext.upper())

def mono_decrypt(ciphertext: str, key_map: dict) -> str:
    reverse_map = {v: k for k, v in key_map.items()}
    return ''.join(reverse_map.get(c, c) for c in ciphertext.upper())


# =============================================================================
# 3. POLYALPHABETIC CIPHER — Vigenère  (Substitution)
# =============================================================================
# Uses a repeating keyword; each keyword letter defines a different Caesar shift.

def vigenere_encrypt(plaintext: str, key: str) -> str:
    result, key_idx = [], 0
    key = key.upper()
    for ch in plaintext.upper():
        if ch.isalpha():
            shift = ord(key[key_idx % len(key)]) - ord('A')
            result.append(chr((ord(ch) - ord('A') + shift) % 26 + ord('A')))
            key_idx += 1
        else:
            result.append(ch)
    return ''.join(result)

def vigenere_decrypt(ciphertext: str, key: str) -> str:
    result, key_idx = [], 0
    key = key.upper()
    for ch in ciphertext.upper():
        if ch.isalpha():
            shift = ord(key[key_idx % len(key)]) - ord('A')
            result.append(chr((ord(ch) - ord('A') - shift) % 26 + ord('A')))
            key_idx += 1
        else:
            result.append(ch)
    return ''.join(result)


# =============================================================================
# 4. PLAYFAIR CIPHER  (Substitution)
# =============================================================================
# Encrypts digrams (letter pairs) using a 5×5 key matrix (I/J merged).

def playfair_generate_matrix(key: str) -> list:
    key = key.upper().replace('J', 'I')
    seen, order = set(), []
    for ch in key + string.ascii_uppercase:
        if ch == 'J':
            continue
        if ch not in seen:
            seen.add(ch)
            order.append(ch)
    return [order[i*5:(i+1)*5] for i in range(5)]

def _playfair_find(matrix, ch):
    for r, row in enumerate(matrix):
        if ch in row:
            return r, row.index(ch)
    return None, None

def _playfair_prepare(text: str) -> list:
    text   = text.upper().replace('J', 'I').replace(' ', '')
    pairs  = []
    i      = 0
    while i < len(text):
        a = text[i]
        b = text[i+1] if i+1 < len(text) else 'X'
        if a == b:
            pairs.append((a, 'X'))
            i += 1
        else:
            pairs.append((a, b))
            i += 2
    return pairs

def playfair_encrypt(plaintext: str, key: str) -> str:
    matrix = playfair_generate_matrix(key)
    pairs  = _playfair_prepare(plaintext)
    result = []
    for a, b in pairs:
        r1, c1 = _playfair_find(matrix, a)
        r2, c2 = _playfair_find(matrix, b)
        if r1 == r2:                          # same row → shift right
            result += [matrix[r1][(c1+1)%5], matrix[r2][(c2+1)%5]]
        elif c1 == c2:                        # same col → shift down
            result += [matrix[(r1+1)%5][c1], matrix[(r2+1)%5][c2]]
        else:                                 # rectangle swap
            result += [matrix[r1][c2], matrix[r2][c1]]
    return ''.join(result)

def playfair_decrypt(ciphertext: str, key: str) -> str:
    matrix = playfair_generate_matrix(key)
    text   = ciphertext.upper()
    pairs  = [(text[i], text[i+1]) for i in range(0, len(text), 2)]
    result = []
    for a, b in pairs:
        r1, c1 = _playfair_find(matrix, a)
        r2, c2 = _playfair_find(matrix, b)
        if r1 == r2:
            result += [matrix[r1][(c1-1)%5], matrix[r2][(c2-1)%5]]
        elif c1 == c2:
            result += [matrix[(r1-1)%5][c1], matrix[(r2-1)%5][c2]]
        else:
            result += [matrix[r1][c2], matrix[r2][c1]]
    return ''.join(result)


# =============================================================================
# 5. HILL CIPHER  (Substitution)
# =============================================================================
# Encrypts n-letter blocks using matrix multiplication over Z₂₆.

def hill_encrypt(plaintext: str, key_matrix: list) -> str:
    n       = len(key_matrix)
    K       = np.array(key_matrix) % 26
    text    = plaintext.upper().replace(' ', '')
    # Pad to multiple of n
    while len(text) % n:
        text += 'X'
    result = []
    for i in range(0, len(text), n):
        block  = np.array([ord(c) - ord('A') for c in text[i:i+n]])
        enc    = (K @ block) % 26
        result += [chr(int(v) + ord('A')) for v in enc]
    return ''.join(result)

def _matrix_mod_inverse(matrix, mod=26):
    """Compute modular inverse of a matrix using numpy + adjugate method."""
    det = int(round(np.linalg.det(matrix))) % mod
    # Find multiplicative inverse of determinant mod 26
    det_inv = None
    for i in range(1, mod):
        if (det * i) % mod == 1:
            det_inv = i
            break
    if det_inv is None:
        raise ValueError("Key matrix is not invertible mod 26")
    n      = matrix.shape[0]
    adj    = np.zeros((n, n), dtype=int)
    for r in range(n):
        for c in range(n):
            minor      = np.delete(np.delete(matrix, r, axis=0), c, axis=1)
            cofactor   = int(round(np.linalg.det(minor))) * ((-1) ** (r + c))
            adj[c][r]  = cofactor % mod
    return (det_inv * adj) % mod

def hill_decrypt(ciphertext: str, key_matrix: list) -> str:
    n       = len(key_matrix)
    K       = np.array(key_matrix) % 26
    K_inv   = _matrix_mod_inverse(K)
    text    = ciphertext.upper().replace(' ', '')
    result  = []
    for i in range(0, len(text), n):
        block  = np.array([ord(c) - ord('A') for c in text[i:i+n]])
        dec    = (K_inv @ block) % 26
        result += [chr(int(v) + ord('A')) for v in dec]
    return ''.join(result)


# =============================================================================
# 6. RAIL FENCE CIPHER  (Transposition)
# =============================================================================
# Writes the text in a zigzag pattern across 'rails' then reads row by row.

def rail_fence_encrypt(plaintext: str, rails: int) -> str:
    fence = [[] for _ in range(rails)]
    rail, direction = 0, 1
    for ch in plaintext:
        fence[rail].append(ch)
        if rail == 0:
            direction = 1
        elif rail == rails - 1:
            direction = -1
        rail += direction
    return ''.join(''.join(r) for r in fence)

def rail_fence_decrypt(ciphertext: str, rails: int) -> str:
    n       = len(ciphertext)
    pattern = []
    rail, direction = 0, 1
    for i in range(n):
        pattern.append(rail)
        if rail == 0:
            direction = 1
        elif rail == rails - 1:
            direction = -1
        rail += direction

    indices = sorted(range(n), key=lambda i: pattern[i])
    result  = [''] * n
    for pos, ch in zip(indices, ciphertext):
        result[pos] = ch
    return ''.join(result)


# =============================================================================
# 7. ROW TRANSPOSITION CIPHER  (Transposition)
# =============================================================================
# Arranges text into rows; columns are then read in the order defined by the key.

def row_transposition_encrypt(plaintext: str, key: str) -> str:
    key      = key.upper()
    n_cols   = len(key)
    text     = plaintext.upper().replace(' ', '')
    # Pad with 'X' to fill the grid
    while len(text) % n_cols:
        text += 'X'
    n_rows   = len(text) // n_cols
    grid     = [list(text[i*n_cols:(i+1)*n_cols]) for i in range(n_rows)]
    # Determine column read order by sorting key alphabetically
    order    = sorted(range(n_cols), key=lambda i: key[i])
    result   = []
    for col in order:
        for row in grid:
            result.append(row[col])
    return ''.join(result)

def row_transposition_decrypt(ciphertext: str, key: str) -> str:
    key      = key.upper()
    n_cols   = len(key)
    n_rows   = len(ciphertext) // n_cols
    order    = sorted(range(n_cols), key=lambda i: key[i])
    # Rebuild each column
    cols     = {}
    idx      = 0
    for col in order:
        cols[col] = list(ciphertext[idx:idx+n_rows])
        idx += n_rows
    # Read off row by row
    result = []
    for r in range(n_rows):
        for c in range(n_cols):
            result.append(cols[c][r])
    return ''.join(result)


# =============================================================================
# 8. PRODUCT CIPHER  (Transposition + Substitution combined)
# =============================================================================
# Applies multiple ciphers in sequence (here: Vigenère → Row Transposition).
# Product ciphers are the conceptual foundation of modern block ciphers (AES).

def product_cipher_encrypt(plaintext: str, vigenere_key: str, transposition_key: str) -> str:
    step1 = vigenere_encrypt(plaintext, vigenere_key)           # Substitution
    step2 = row_transposition_encrypt(step1, transposition_key) # Transposition
    return step2

def product_cipher_decrypt(ciphertext: str, vigenere_key: str, transposition_key: str) -> str:
    step1 = row_transposition_decrypt(ciphertext, transposition_key)  # Undo transposition
    step2 = vigenere_decrypt(step1, vigenere_key)                     # Undo substitution
    return step2

In [ ]:
def run_interactive_menu():
    while True:
        print("\n--- Interactive Cipher Menu ---")
        print("1. Caesar Cipher")
        print("2. Monoalphabetic Cipher")
        print("3. Vigenère Cipher")
        print("4. Playfair Cipher")
        print("5. Hill Cipher")
        print("6. Rail Fence Cipher")
        print("7. Row Transposition Cipher")
        print("8. Product Cipher")
        print("0. Exit")

        choice = input("Enter your choice (0-8): ")

        if choice == '0':
            print("Exiting cipher menu. Goodbye!")
            break

        try:
            cipher_num = int(choice)
            if not (0 < cipher_num <= 8):
                raise ValueError
        except ValueError:
            print("Invalid choice. Please enter a number between 0 and 8.")
            continue

        mode = input("Do you want to (e)ncrypt or (d)ecrypt? ").lower()
        if mode not in ['e', 'd']:
            print("Invalid mode. Please enter 'e' for encrypt or 'd' for decrypt.")
            continue

        text = input(f"Enter the {'plaintext' if mode == 'e' else 'ciphertext'}: ")
        output = ""

        try:
            if cipher_num == 1: # Caesar Cipher
                shift = int(input("Enter the shift value (integer): "))
                if mode == 'e':
                    output = caesar_encrypt(text, shift)
                else:
                    output = caesar_decrypt(text, shift)

            elif cipher_num == 2: # Monoalphabetic Cipher
                keyword = input("Enter the keyword: ")
                key_map = mono_generate_key(keyword)
                if mode == 'e':
                    output = mono_encrypt(text, key_map)
                else:
                    output = mono_decrypt(text, key_map)

            elif cipher_num == 3: # Vigenère Cipher
                key = input("Enter the key: ")
                if mode == 'e':
                    output = vigenere_encrypt(text, key)
                else:
                    output = vigenere_decrypt(text, key)

            elif cipher_num == 4: # Playfair Cipher
                key = input("Enter the key: ")
                if mode == 'e':
                    output = playfair_encrypt(text, key)
                else:
                    output = playfair_decrypt(text, key)

            elif cipher_num == 5: # Hill Cipher
                key_size = int(input("Enter the size of the key matrix (e.g., 2 for 2x2, 3 for 3x3): "))
                print(f"Enter the {key_size}x{key_size} key matrix row by row, space-separated integers:")
                key_matrix = []
                for i in range(key_size):
                    row = list(map(int, input(f"Row {i+1}: ").split()))
                    if len(row) != key_size:
                        raise ValueError(f"Row {i+1} must have {key_size} integers.")
                    key_matrix.append(row)

                if mode == 'e':
                    output = hill_encrypt(text, key_matrix)
                else:
                    output = hill_decrypt(text, key_matrix)

            elif cipher_num == 6: # Rail Fence Cipher
                rails = int(input("Enter the number of rails (integer): "))
                if mode == 'e':
                    output = rail_fence_encrypt(text, rails)
                else:
                    output = rail_fence_decrypt(text, rails)

            elif cipher_num == 7: # Row Transposition Cipher
                key = input("Enter the key: ")
                if mode == 'e':
                    output = row_transposition_encrypt(text, key)
                else:
                    output = row_transposition_decrypt(text, key)

            elif cipher_num == 8: # Product Cipher
                vigenere_key = input("Enter the Vigenère key: ")
                transposition_key = input("Enter the Row Transposition key: ")
                if mode == 'e':
                    output = product_cipher_encrypt(text, vigenere_key, transposition_key)
                else:
                    output = product_cipher_decrypt(text, vigenere_key, transposition_key)

            print(f"Result: {output}")

        except Exception as e:
            print(f"An error occurred: {e}")


run_interactive_menu()